# Notebook 4: Cost-Complexity Pruning — Sculpting the Optimal Tree

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Theme:** House Price Prediction (binary classification: above/below median) + regression  
**Series:** Week 9 — Tree-Based Methods & Gradient Boosting

---

## Why This Matters

Notebook 3 showed how pre-pruning hyperparameters (`max_depth`, `min_samples_leaf`) shape a tree **while it grows**. But these blunt tools apply the same constraint uniformly across all branches — they may stop one branch too early while letting another branch over-grow.

**Post-pruning** takes the opposite approach:
1. Grow the **full, unconstrained tree** first
2. Then selectively **remove** branches whose complexity cost exceeds their benefit

Cost-Complexity Pruning is sklearn's built-in post-pruning algorithm. Understanding it gives you insight into how the bias–variance trade-off is formalised mathematically.

---

## Real-World Analogy First: The Marble Sculptor

A sculptor begins with a raw block of marble and makes a detailed sculpture:

- **First pass (grow full tree):** Carve every detail — every eyelash, every fold of cloth, every vein on the hand. The result is technically perfect... but overly detailed. It looks like it was traced from a photograph of one specific person (one specific training set).
- **Post-pruning:** Now step back. For each detail: *"Does this eyelash add enough artistic value per chisel stroke to justify keeping it?"* If the answer is no — remove it. The result is a cleaner sculpture that captures the **essence** of the subject, not the noise.

The sculptor's "artistic value per chisel stroke" is exactly the **effective alpha** in cost-complexity pruning: the improvement in training error per unit of complexity added.

> **Key insight:** Post-pruning lets each branch be evaluated on its own merits. A branch in a dense, informative region of feature space will survive; a branch that exists only to fit noise will not.

## The Mathematics of Cost-Complexity Pruning

### Objective

For a tree $T$, define the **cost-complexity criterion**:

$$R_\alpha(T) = R(T) + \alpha \cdot |T|$$

where:
- $R(T)$ = total training impurity (MSE for regression, weighted misclassification for classification)
- $|T|$ = number of leaf nodes (complexity term)
- $\alpha \geq 0$ = complexity penalty parameter (the tuning knob)

### The Pruning Path

As $\alpha$ increases from 0, the algorithm generates a sequence of nested subtrees:

$$T_{\max} \supseteq T_1 \supseteq T_2 \supseteq \cdots \supseteq T_1 \text{(root only)}$$

Each transition removes the **weakest link** — the internal node with the smallest **effective alpha**:

$$\tilde{\alpha}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}$$

where:
- $R(t)$ = impurity if node $t$ is made a leaf
- $R(T_t)$ = total impurity of the subtree rooted at $t$
- $|T_t|$ = number of leaves in subtree rooted at $t$

**Interpretation of effective alpha:** It is the *improvement in training error per leaf added* by keeping this subtree rather than replacing it with a single leaf. A small $\tilde{\alpha}(t)$ means this subtree barely improves training error while adding many leaves — it is the most cost-inefficient subtree to keep.

### Selecting the Best $\alpha$

- Computing the pruning path uses **training data only** (to calculate impurities)
- Selecting the best $\alpha$ uses **validation data or cross-validation** (to measure generalization)
- These **must** be separate datasets

### Pre-Pruning vs Post-Pruning

| Aspect | Pre-Pruning | Post-Pruning |
|---|---|---|
| When applied | During growth | After full tree grown |
| Control lever | max_depth, min_samples_* | alpha |
| Branch sensitivity | Uniform across tree | Per-branch (adaptive) |
| Risk | Stops too early in some branches | Computationally heavier |
| Typical result | Good with tuning | Often better generalization |

## Section 1: Setup and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

In [ ]:
# --- Synthetic House Price Dataset: 800 samples, 8 features ---
np.random.seed(42)
N = 800

sqft         = np.random.normal(1800, 500, N).clip(500, 4000)
bedrooms     = np.random.randint(1, 7, N).astype(float)
bathrooms    = np.random.uniform(1, 4, N)
lot_size     = np.random.normal(7000, 2000, N).clip(1000, 20000)
age          = np.random.randint(0, 80, N).astype(float)
garage       = np.random.randint(0, 4, N).astype(float)
school_score = np.random.uniform(3, 10, N)
dist_center  = np.random.uniform(1, 30, N)

price = (
    150 * sqft / 1000
    + 5000 * bedrooms
    + 8000 * bathrooms
    + 2 * lot_size
    - 500 * age
    + 4000 * garage
    + 3000 * school_score
    - 1500 * dist_center
    + np.random.normal(0, 15000, N)
    + 80000
).clip(50000, 800000)

feature_names = ['sqft', 'bedrooms', 'bathrooms', 'lot_size', 'age', 'garage', 'school_score', 'dist_center']

df = pd.DataFrame({
    'sqft': sqft, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
    'lot_size': lot_size, 'age': age, 'garage': garage,
    'school_score': school_score, 'dist_center': dist_center,
    'price': price
})

median_price = df['price'].median()
df['above_median'] = (df['price'] > median_price).astype(int)

print(f"Dataset: {df.shape[0]} samples, {len(feature_names)} features")
print(f"Price range: ${df['price'].min():,.0f} – ${df['price'].max():,.0f}")
print(f"Median price: ${median_price:,.0f}")
print(f"Class balance: {df['above_median'].mean():.1%} above median")

In [ ]:
# Train / Validation / Test Split: 60 / 20 / 20
X = df[feature_names].values
y_clf = df['above_median'].values
y_reg = df['price'].values

X_temp, X_test, y_temp_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, random_state=42)
X_train, X_val, y_train_c, y_val_c = train_test_split(X_temp, y_temp_c, test_size=0.25, random_state=42)

_, _, y_temp_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_r, y_val_r = train_test_split(X_temp, y_temp_r, test_size=0.25, random_state=42)

print(f"Train:      {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test:       {X_test.shape[0]} samples")

## Section 2: Grow the Full (Unpruned) Tree — Demonstrating Overfitting

In [ ]:
# Grow unconstrained tree (no max_depth, no min_samples limits beyond sklearn defaults)
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train_c)

train_acc_full = accuracy_score(y_train_c, full_tree.predict(X_train))
val_acc_full   = accuracy_score(y_val_c,   full_tree.predict(X_val))
test_acc_full  = accuracy_score(y_test_c,  full_tree.predict(X_test))

n_nodes_full  = full_tree.tree_.node_count
n_leaves_full = full_tree.get_n_leaves()
depth_full    = full_tree.get_depth()

print("=" * 50)
print("Full (Unpruned) Tree Statistics")
print("=" * 50)
print(f"Total nodes:    {n_nodes_full}")
print(f"Leaf nodes:     {n_leaves_full}")
print(f"Internal nodes: {n_nodes_full - n_leaves_full}")
print(f"Max depth:      {depth_full}")
print()
print(f"Train accuracy: {train_acc_full:.4f}  ({train_acc_full:.1%})")
print(f"Val accuracy:   {val_acc_full:.4f}  ({val_acc_full:.1%})")
print(f"Test accuracy:  {test_acc_full:.4f}  ({test_acc_full:.1%})")
print()
overfit_gap = train_acc_full - val_acc_full
print(f"Overfitting gap (train - val): {overfit_gap:.1%}")
print()
if train_acc_full > 0.99:
    print(">>> The full tree has memorised the training set (near-perfect train accuracy).")
    print(f">>> Validation accuracy is {val_acc_full:.1%} — clear evidence of overfitting.")

In [ ]:
# Visualise a small portion of the full tree (depth 3 only — it's too large to show fully)
fig, ax = plt.subplots(figsize=(22, 7))
plot_tree(
    full_tree,
    max_depth=3,
    feature_names=feature_names,
    class_names=['Below Median', 'Above Median'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
ax.set_title(
    f"Full Tree (first 3 of {depth_full} levels shown)\n"
    f"Total: {n_nodes_full} nodes, {n_leaves_full} leaves — "
    f"Train={train_acc_full:.1%}, Val={val_acc_full:.1%}",
    fontsize=11
)
plt.tight_layout()
plt.show()
print(f"\nNote: This shows only 3 of {depth_full} depth levels. The full tree has {n_leaves_full} leaves.")

## Section 3: Computing the Pruning Path

sklearn's `cost_complexity_pruning_path()` returns the sequence of `(alpha, impurity)` pairs that define the pruning trajectory. Each alpha value in the path corresponds to the moment when a specific branch gets pruned.

In [ ]:
# Compute pruning path
path = full_tree.cost_complexity_pruning_path(X_train, y_train_c)
ccp_alphas = path.ccp_alphas   # shape: (n_pruning_steps,)
impurities = path.impurities   # total impurity at each alpha

print(f"Number of pruning steps: {len(ccp_alphas)}")
print(f"Alpha range: [{ccp_alphas[0]:.6f}, {ccp_alphas[-1]:.6f}]")
print()
print("First 10 pruning steps:")
print(f"  {'Step':>5}  {'Alpha':>12}  {'Impurity':>10}")
print("  " + "-" * 32)
for i, (a, imp) in enumerate(zip(ccp_alphas[:10], impurities[:10])):
    print(f"  {i:>5}  {a:>12.6f}  {imp:>10.6f}")
print("  ...")
print()
print("Last 5 pruning steps:")
for i, (a, imp) in enumerate(zip(ccp_alphas[-5:], impurities[-5:])):
    idx = len(ccp_alphas) - 5 + i
    print(f"  {idx:>5}  {a:>12.6f}  {imp:>10.6f}")
print(f"\n(alpha={ccp_alphas[-1]:.6f} = root-only tree, maximum impurity)")

In [ ]:
# Visualise the pruning path
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(ccp_alphas[:-1], impurities[:-1], 'o-', color='steelblue', markersize=3, linewidth=1.5)
ax.set_xlabel('Effective Alpha (ccp_alpha)')
ax.set_ylabel('Total Training Impurity')
ax.set_title('Pruning Path: Impurity vs Alpha')
ax.set_xscale('log' if ccp_alphas[1] > 0 else 'linear')

ax = axes[1]
# Show distribution of effective alphas (log scale)
valid_alphas = ccp_alphas[(ccp_alphas > 0) & (ccp_alphas < ccp_alphas[-1])]
ax.hist(np.log10(valid_alphas + 1e-10), bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xlabel('log10(Effective Alpha)')
ax.set_ylabel('Number of Branches')
ax.set_title('Distribution of Effective Alphas\n(each bar = one branch being pruned)')

plt.suptitle('The Pruning Path: From Full Tree to Root', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Each alpha value = one branch gets pruned. Moving left-to-right: tree simplifies.")

## Section 4: Fit Trees Along the Pruning Path

For each alpha value in the path, we fit a new `DecisionTreeClassifier` with that `ccp_alpha`. This is how sklearn implements post-pruning — you specify the penalty, and sklearn internally prunes to the optimal subtree for that penalty.

In [ ]:
# Fit a tree for each alpha in the pruning path
# (Exclude the last alpha — it gives a root-only tree with no predictive power)
# Subsample alphas for speed (keep every step but limit to 80 trees)
n_alphas = len(ccp_alphas)
if n_alphas > 80:
    idx_keep = np.unique(np.linspace(0, n_alphas - 2, 80, dtype=int))
else:
    idx_keep = np.arange(n_alphas - 1)

selected_alphas = ccp_alphas[idx_keep]

train_accs, val_accs = [], []
node_counts, leaf_counts, depths_list = [], [], []

for alpha in selected_alphas:
    clf = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    clf.fit(X_train, y_train_c)
    train_accs.append(accuracy_score(y_train_c, clf.predict(X_train)))
    val_accs.append(accuracy_score(y_val_c,     clf.predict(X_val)))
    node_counts.append(clf.tree_.node_count)
    leaf_counts.append(clf.get_n_leaves())
    depths_list.append(clf.get_depth())

best_alpha_idx = np.argmax(val_accs)
best_alpha = selected_alphas[best_alpha_idx]

print(f"Number of trees fit along pruning path: {len(selected_alphas)}")
print(f"Best ccp_alpha (by val accuracy): {best_alpha:.6f}")
print(f"  Val accuracy at best alpha:  {val_accs[best_alpha_idx]:.4f} ({val_accs[best_alpha_idx]:.1%})")
print(f"  Train accuracy at best alpha: {train_accs[best_alpha_idx]:.4f} ({train_accs[best_alpha_idx]:.1%})")
print(f"  Nodes at best alpha:         {node_counts[best_alpha_idx]}")
print(f"  Leaves at best alpha:        {leaf_counts[best_alpha_idx]}")
print(f"  Depth at best alpha:         {depths_list[best_alpha_idx]}")

## Section 5: Three-Panel Visualisation — The Pruning Story

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel 1: Node count vs alpha ---
ax = axes[0]
ax.plot(selected_alphas, node_counts, 'o-', color='purple', markersize=3, linewidth=1.8)
ax.axvline(best_alpha, color='darkgreen', linestyle='--', linewidth=2,
           label=f'Best alpha={best_alpha:.5f}')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Number of Nodes')
ax.set_title('Tree Size vs Alpha\n(nodes decrease as alpha increases)')
ax.set_xscale('symlog', linthresh=1e-4)
ax.legend(fontsize=8)
ax.annotate(f'Full tree\n{n_nodes_full} nodes', xy=(selected_alphas[0], node_counts[0]),
            xytext=(selected_alphas[0]*5, node_counts[0]*0.85),
            fontsize=8, color='purple',
            arrowprops=dict(arrowstyle='->', color='purple', lw=1.2))

# --- Panel 2: Train accuracy vs alpha ---
ax = axes[1]
ax.plot(selected_alphas, train_accs, 'o-', color='steelblue', markersize=3, linewidth=1.8,
        label='Train Accuracy')
ax.axvline(best_alpha, color='darkgreen', linestyle='--', linewidth=2)
ax.axhline(train_acc_full, color='steelblue', linestyle=':', alpha=0.5,
           label=f'Full tree train ({train_acc_full:.1%})')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Accuracy')
ax.set_title('Train Accuracy vs Alpha\n(initially falls as complexity removed)')
ax.set_xscale('symlog', linthresh=1e-4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(fontsize=8)

# --- Panel 3: Val accuracy vs alpha ---
ax = axes[2]
ax.plot(selected_alphas, val_accs, 's-', color='tomato', markersize=3, linewidth=1.8,
        label='Val Accuracy')
ax.axvline(best_alpha, color='darkgreen', linestyle='--', linewidth=2,
           label=f'Optimal alpha')
ax.axhline(val_acc_full, color='tomato', linestyle=':', alpha=0.5,
           label=f'Full tree val ({val_acc_full:.1%})')
ax.scatter([best_alpha], [val_accs[best_alpha_idx]], s=120, color='darkgreen',
           zorder=5, label=f'Best val={val_accs[best_alpha_idx]:.1%}')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Accuracy')
ax.set_title('Val Accuracy vs Alpha\n(improves then degrades — find the peak)')
ax.set_xscale('symlog', linthresh=1e-4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(fontsize=8)

plt.suptitle('Cost-Complexity Pruning Path: Three Views', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Reading the plots:")
print("  Panel 1: As alpha increases, tree size drops (branches get pruned)")
print("  Panel 2: Train accuracy falls — pruning removes branches that fit training data")
print("  Panel 3: Val accuracy IMPROVES initially (removing noise) then falls (removing signal)")
print(f"  The sweet spot: alpha={best_alpha:.5f}, val accuracy={val_accs[best_alpha_idx]:.1%}")

## Section 6: Finding Optimal Alpha via Cross-Validation

Using a single validation set to choose alpha can be noisy. Cross-validation gives a more reliable estimate of which alpha generalizes best.

In [ ]:
# Recompute pruning path on full train+val data for CV
X_cv = np.vstack([X_train, X_val])
y_cv_c = np.concatenate([y_train_c, y_val_c])

# Get pruning path on combined data
cv_path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_cv, y_cv_c)
cv_alphas = cv_path.ccp_alphas[:-1]  # drop last (root-only)

# Subsample for speed
if len(cv_alphas) > 50:
    cv_alpha_idx = np.unique(np.linspace(0, len(cv_alphas)-1, 50, dtype=int))
    cv_alphas_sel = cv_alphas[cv_alpha_idx]
else:
    cv_alphas_sel = cv_alphas

cv_means, cv_stds = [], []

for alpha in cv_alphas_sel:
    clf = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(clf, X_cv, y_cv_c, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds  = np.array(cv_stds)
best_cv_idx   = np.argmax(cv_means)
best_cv_alpha = cv_alphas_sel[best_cv_idx]

print(f"CV optimal alpha:        {best_cv_alpha:.6f}")
print(f"CV mean accuracy:        {cv_means[best_cv_idx]:.4f} ± {cv_stds[best_cv_idx]:.4f}")
print(f"Val-set optimal alpha:   {best_alpha:.6f}")
print(f"Val-set accuracy:        {val_accs[best_alpha_idx]:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cv_alphas_sel, cv_means, 'o-', color='steelblue', linewidth=2, markersize=4, label='CV Mean Accuracy')
ax.fill_between(cv_alphas_sel,
                cv_means - cv_stds,
                cv_means + cv_stds,
                alpha=0.2, color='steelblue', label='±1 std')
ax.axvline(best_cv_alpha, color='darkgreen', linestyle='--', linewidth=2,
           label=f'CV optimal alpha={best_cv_alpha:.5f}')
ax.axvline(best_alpha, color='tomato', linestyle=':', linewidth=1.8,
           label=f'Val-set optimal alpha={best_alpha:.5f}')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Accuracy')
ax.set_title('Cross-Validation Accuracy vs ccp_alpha (5-fold)')
ax.set_xscale('symlog', linthresh=1e-4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## Section 7: Compare Unpruned vs Pruned Tree

In [ ]:
# Fit final pruned tree using CV-optimal alpha
pruned_tree = DecisionTreeClassifier(ccp_alpha=best_cv_alpha, random_state=42)
pruned_tree.fit(X_train, y_train_c)

# Metrics comparison
metrics = {}
for name, tree in [('Unpruned (full)', full_tree), ('Pruned (CV-optimal)', pruned_tree)]:
    metrics[name] = {
        'train_acc':  accuracy_score(y_train_c, tree.predict(X_train)),
        'val_acc':    accuracy_score(y_val_c,   tree.predict(X_val)),
        'test_acc':   accuracy_score(y_test_c,  tree.predict(X_test)),
        'n_nodes':    tree.tree_.node_count,
        'n_leaves':   tree.get_n_leaves(),
        'depth':      tree.get_depth()
    }

print("=" * 65)
print(f"{'Metric':<25} {'Unpruned':>18} {'Pruned':>18}")
print("=" * 65)
labels = {
    'train_acc': 'Train Accuracy',
    'val_acc':   'Val Accuracy',
    'test_acc':  'Test Accuracy',
    'n_nodes':   'Total Nodes',
    'n_leaves':  'Leaf Nodes',
    'depth':     'Max Depth'
}
for key, label in labels.items():
    u = metrics['Unpruned (full)'][key]
    p = metrics['Pruned (CV-optimal)'][key]
    if key in ('train_acc', 'val_acc', 'test_acc'):
        print(f"{label:<25} {u:>17.1%} {p:>17.1%}")
    else:
        print(f"{label:<25} {u:>18} {p:>18}")
print("=" * 65)

size_reduction = (1 - metrics['Pruned (CV-optimal)']['n_nodes'] / metrics['Unpruned (full)']['n_nodes']) * 100
acc_change     = (metrics['Pruned (CV-optimal)']['test_acc'] - metrics['Unpruned (full)']['test_acc']) * 100
print(f"\nTree size reduction: {size_reduction:.1f}%")
print(f"Test accuracy change: {acc_change:+.1f} percentage points")

In [ ]:
# Side-by-side tree visualisation
fig, axes = plt.subplots(1, 2, figsize=(24, 8))

plot_tree(
    full_tree,
    max_depth=3,
    feature_names=feature_names,
    class_names=['Below', 'Above'],
    filled=True, rounded=True, fontsize=7.5,
    ax=axes[0]
)
axes[0].set_title(
    f"Unpruned Tree (first 3 of {full_tree.get_depth()} levels)\n"
    f"{full_tree.tree_.node_count} nodes, {full_tree.get_n_leaves()} leaves\n"
    f"Train={metrics['Unpruned (full)']['train_acc']:.1%}, "
    f"Val={metrics['Unpruned (full)']['val_acc']:.1%}, "
    f"Test={metrics['Unpruned (full)']['test_acc']:.1%}",
    fontsize=10
)

pruned_depth = pruned_tree.get_depth()
show_depth   = min(pruned_depth, 4)
plot_tree(
    pruned_tree,
    max_depth=show_depth,
    feature_names=feature_names,
    class_names=['Below', 'Above'],
    filled=True, rounded=True, fontsize=7.5,
    ax=axes[1]
)
axes[1].set_title(
    f"Pruned Tree (ccp_alpha={best_cv_alpha:.5f}, showing {show_depth} of {pruned_depth} levels)\n"
    f"{pruned_tree.tree_.node_count} nodes, {pruned_tree.get_n_leaves()} leaves\n"
    f"Train={metrics['Pruned (CV-optimal)']['train_acc']:.1%}, "
    f"Val={metrics['Pruned (CV-optimal)']['val_acc']:.1%}, "
    f"Test={metrics['Pruned (CV-optimal)']['test_acc']:.1%}",
    fontsize=10
)

plt.suptitle('Unpruned vs Pruned Decision Tree', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 8: Regression Pruning — House Price Prediction

The same cost-complexity framework applies to regression trees (using MSE as the impurity measure). Let's see how pruning affects prediction of the actual price (in dollars).

In [ ]:
# Grow full regression tree
full_reg_tree = DecisionTreeRegressor(random_state=42)
full_reg_tree.fit(X_train, y_train_r)

train_rmse_full = np.sqrt(mean_squared_error(y_train_r, full_reg_tree.predict(X_train)))
val_rmse_full   = np.sqrt(mean_squared_error(y_val_r,   full_reg_tree.predict(X_val)))

print(f"Full regression tree: {full_reg_tree.tree_.node_count} nodes, {full_reg_tree.get_n_leaves()} leaves")
print(f"  Train RMSE: ${train_rmse_full:,.0f}")
print(f"  Val RMSE:   ${val_rmse_full:,.0f}")
print(f"  Overfitting: val/train RMSE ratio = {val_rmse_full/train_rmse_full:.2f}x")

# Compute regression pruning path
reg_path    = full_reg_tree.cost_complexity_pruning_path(X_train, y_train_r)
reg_alphas  = reg_path.ccp_alphas

# Subsample for speed
if len(reg_alphas) > 80:
    reg_idx_keep = np.unique(np.linspace(0, len(reg_alphas) - 2, 80, dtype=int))
else:
    reg_idx_keep = np.arange(len(reg_alphas) - 1)

reg_alphas_sel = reg_alphas[reg_idx_keep]
reg_train_rmse, reg_val_rmse = [], []
reg_n_leaves = []

for alpha in reg_alphas_sel:
    reg = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    reg.fit(X_train, y_train_r)
    reg_train_rmse.append(np.sqrt(mean_squared_error(y_train_r, reg.predict(X_train))))
    reg_val_rmse.append(  np.sqrt(mean_squared_error(y_val_r,   reg.predict(X_val))))
    reg_n_leaves.append(reg.get_n_leaves())

reg_best_idx   = np.argmin(reg_val_rmse)
reg_best_alpha = reg_alphas_sel[reg_best_idx]

print(f"\nOptimal regression ccp_alpha: {reg_best_alpha:.2f}")
print(f"  Val RMSE:   ${reg_val_rmse[reg_best_idx]:,.0f}")
print(f"  Train RMSE: ${reg_train_rmse[reg_best_idx]:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: RMSE vs alpha
ax = axes[0]
ax.plot(reg_alphas_sel, np.array(reg_train_rmse)/1000, 'o-', color='steelblue',
        markersize=3, linewidth=1.8, label='Train RMSE')
ax.plot(reg_alphas_sel, np.array(reg_val_rmse)/1000, 's-', color='tomato',
        markersize=3, linewidth=1.8, label='Val RMSE')
ax.axvline(reg_best_alpha, color='darkgreen', linestyle='--', linewidth=2,
           label=f'Optimal alpha={reg_best_alpha:.0f}')
ax.axhline(val_rmse_full/1000, color='tomato', linestyle=':', alpha=0.5,
           label=f'Full tree val (${val_rmse_full/1000:.0f}k)')
ax.scatter([reg_best_alpha], [reg_val_rmse[reg_best_idx]/1000],
           s=120, color='darkgreen', zorder=5,
           label=f'Best val=${reg_val_rmse[reg_best_idx]/1000:.0f}k')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('RMSE ($k)')
ax.set_title('Regression RMSE vs ccp_alpha')
ax.set_xscale('symlog', linthresh=1e3)
ax.legend(fontsize=8)

# Right: Number of leaves vs alpha
ax = axes[1]
ax.plot(reg_alphas_sel, reg_n_leaves, 'o-', color='purple',
        markersize=3, linewidth=1.8)
ax.axvline(reg_best_alpha, color='darkgreen', linestyle='--', linewidth=2,
           label=f'Optimal alpha={reg_best_alpha:.0f}')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Number of Leaves')
ax.set_title('Regression Tree Size vs ccp_alpha')
ax.set_xscale('symlog', linthresh=1e3)
ax.legend()
ax.annotate(f'{reg_n_leaves[0]} leaves\n(full tree)', (reg_alphas_sel[0], reg_n_leaves[0]),
            xytext=(reg_alphas_sel[5], reg_n_leaves[0]*0.75),
            fontsize=8, color='purple',
            arrowprops=dict(arrowstyle='->', color='purple', lw=1.2))

plt.suptitle('Regression Pruning: House Price Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Final comparison
best_reg = DecisionTreeRegressor(ccp_alpha=reg_best_alpha, random_state=42)
best_reg.fit(X_train, y_train_r)

print("\n" + "=" * 60)
print(f"{'Metric':<22} {'Full Tree':>16} {'Pruned Tree':>16}")
print("=" * 60)
for name, tree in [('Full', full_reg_tree), ('Pruned', best_reg)]:
    tr_rmse = np.sqrt(mean_squared_error(y_train_r, tree.predict(X_train)))
    vl_rmse = np.sqrt(mean_squared_error(y_val_r,   tree.predict(X_val)))
    ts_rmse = np.sqrt(mean_squared_error(y_test_r,  tree.predict(X_test)))
    vl_r2   = r2_score(y_val_r, tree.predict(X_val))
    if name == 'Full':
        row = (tr_rmse, vl_rmse, ts_rmse, vl_r2,
               tree.tree_.node_count, tree.get_n_leaves())
    else:
        row2 = (tr_rmse, vl_rmse, ts_rmse, vl_r2,
                tree.tree_.node_count, tree.get_n_leaves())

for label, v1, v2, fmt in [
    ('Train RMSE', row[0], row2[0], lambda x: f'${x:,.0f}'),
    ('Val RMSE',   row[1], row2[1], lambda x: f'${x:,.0f}'),
    ('Test RMSE',  row[2], row2[2], lambda x: f'${x:,.0f}'),
    ('Val R²',     row[3], row2[3], lambda x: f'{x:.4f}'),
    ('Nodes',      row[4], row2[4], lambda x: f'{x}'),
    ('Leaves',     row[5], row2[5], lambda x: f'{x}')
]:
    print(f"{label:<22} {fmt(v1):>16} {fmt(v2):>16}")
print("=" * 60)

## Section 9: Feature Importance — Does Pruning Change What the Tree Relies On?

In [ ]:
# Compare feature importances: unpruned vs pruned classification tree
fi_full   = full_tree.feature_importances_
fi_pruned = pruned_tree.feature_importances_

fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Unpruned': fi_full,
    'Pruned':   fi_pruned
}).sort_values('Pruned', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, col, color, title in [
    (axes[0], 'Unpruned', 'tomato',    f'Unpruned Tree ({full_tree.get_n_leaves()} leaves)'),
    (axes[1], 'Pruned',   'steelblue', f'Pruned Tree ({pruned_tree.get_n_leaves()} leaves)')
]:
    ax.barh(fi_df['Feature'], fi_df[col], color=color, edgecolor='white', linewidth=0.8)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importances\n{title}')
    ax.set_xlim(0, max(fi_full.max(), fi_pruned.max()) * 1.1)
    for i, v in enumerate(fi_df[col]):
        ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8.5)

plt.suptitle('Feature Importances: Unpruned vs Pruned', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: Pruning concentrates importance on fewer, stronger features.")
print("Features that appeared important in the full tree due to noise-fitting")
print("often lose importance after pruning.")

## Summary: When to Use Cost-Complexity Pruning

### Decision guide

| Situation | Recommendation |
|---|---|
| Need an interpretable tree with few rules | Use max_depth (pre-pruning) — simple and fast |
| Have noisy data with irregular density | Use cost-complexity pruning — adapts per branch |
| Want best possible single tree | Grow full tree, prune with CV |
| Building Random Forest / GBM | Pre-pruning (max_depth/min_samples) is standard |
| Debugging overfitting in a tree | Check pruning path for structure |

### The Key Workflow

```python
# Step 1: Grow full tree
full = DecisionTreeClassifier().fit(X_train, y_train)

# Step 2: Get pruning path (uses training data only)
path = full.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]

# Step 3: Find best alpha via cross-validation
cv_scores = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a), X_train, y_train, cv=5).mean()
             for a in alphas]
best_alpha = alphas[np.argmax(cv_scores)]

# Step 4: Refit with best alpha
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha).fit(X_train, y_train)
```

### Connection to Regularisation

Cost-complexity pruning is the tree equivalent of:
- **L2 regularisation (Ridge)** in linear regression: $\text{Loss} + \lambda \|\mathbf{w}\|^2$
- **L1 regularisation (Lasso)** in linear regression: $\text{Loss} + \lambda \|\mathbf{w}\|_1$

In all cases: you add a penalty on complexity, and tune the penalty strength with cross-validation.

---
## Self-Check Questions

---

**Question 1:** Why is it often better to grow a full tree and prune, rather than stopping growth early (pre-pruning)?

<details>
<summary>Show Answer</summary>

**Pre-pruning applies the same constraint to every branch uniformly.** For example, `max_depth=5` stops every branch at depth 5 — even branches in dense, informative regions of the feature space where more depth would actually help.

**Post-pruning is adaptive:** Each branch is evaluated on its own merits. A branch in a feature-rich, signal-dense region of the data will have high effective alpha (worth keeping), while a branch that barely improves training impurity per leaf added will have low effective alpha (gets pruned first).

Additionally, pre-pruning can get trapped: early stopping prevents the algorithm from discovering splits that only become useful several levels deeper (known as the "horizon effect"). Growing the full tree first avoids this — the algorithm explores all possibilities, then prunes.

The sculptor analogy: it is easier to remove detail from a finished sculpture than to know in advance exactly which details to never carve.

</details>

---

**Question 2:** The pruning path shows alpha values where each branch gets removed. A branch with effective alpha=0.02 gets pruned before one with effective alpha=0.05. What does this tell you about the relative costs of those two branches?

<details>
<summary>Show Answer</summary>

**The branch with effective alpha=0.02 is less cost-effective to keep.**

Recall the effective alpha formula:
$$\tilde{\alpha}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}$$

This is the *improvement in training error per leaf added* by keeping the subtree at node $t$ instead of a single leaf.

- **Branch with $\tilde{\alpha}=0.02$:** Each additional leaf in this subtree reduces training impurity by only 0.02 units. It is barely worth its complexity cost. Gets pruned first.
- **Branch with $\tilde{\alpha}=0.05$:** Each additional leaf reduces training impurity by 0.05 units — 2.5× more valuable per unit of complexity. Gets pruned later.

In the sculptor analogy: the 0.02-alpha branch is a detail (say, a fingernail) that adds almost no artistic value per chisel stroke. The 0.05-alpha branch (say, a hand) adds much more value per effort and is worth keeping longer.

</details>

---

**Question 3:** Cost-complexity pruning uses training data to compute effective alphas but validation data to select alpha. Why must these be separate datasets?

<details>
<summary>Show Answer</summary>

**To avoid optimistic bias / data leakage in two distinct ways:**

**Why training data for effective alphas?**
Effective alphas measure how much a branch is fitting the *training signal*. This is intentional — we want to rank branches by their training usefulness. The training data defines the pruning path; it tells us which branches are "weak links" in terms of fitting the data they were grown on.

**Why validation/held-out data for selecting alpha?**
The goal of selecting alpha is to find the penalty level that gives best *generalization* — performance on unseen data. If we used the same training data to both:
1. Compute the pruning path, AND
2. Evaluate which alpha is best

...then higher alpha (simpler trees) would always look worse because we'd be measuring training performance, not generalization. The training set "prefers" the full tree — it was grown to minimize training impurity.

By evaluating on validation/held-out data, we get an unbiased estimate of which tree complexity generalizes best to new data. This is the same principle behind any train/validation split: you tune on one, evaluate on the other.

**Concretely:** Using the same data for both steps would be like a student writing their own exam questions AND grading their own exam. The separation ensures objectivity.

</details>